# Cs 6S-to-60P ARC / rydcalc laser-power comparison

Goal: compare ARC and the local `rydcalc` submodule on the same alkali transition:

- atom: `133Cs`
- transition: `|6S1/2, mj=1/2> -> |60P3/2, mj=3/2>`
- polarization: `sigma+`, so `q = +1`
- beam model: circular Gaussian beam at the atom, `w_x = w_y = 10 um`
- reported powers: target `Omega/(2 pi) = 1, 5, 10 MHz`

This notebook uses ARC as the alkali reference and uses `rydcalc.Cesium133` as the implementation under test. The comparison is intentionally direct: both packages are asked for the same fine-structure, `mj`-resolved electric-dipole matrix element.


## Formula conventions

For a single-photon electric-dipole transition at the beam center,

```text
Omega = |d_eff| E0 / hbar
I0 = 2 P / (pi w_x w_y)
I0 = c epsilon0 E0^2 / 2
```

Solving for optical power in the Gaussian beam gives

```text
P = pi w_x w_y c epsilon0 hbar^2 Omega^2 / (4 |d_eff|^2)
```

Both ARC and rydcalc return electric-dipole matrix elements in atomic units `e a0`. The SI conversion is

```text
d_SI = d_au * e * a0
```

The sign of an electric-dipole matrix element depends on phase conventions. The power calculation therefore uses `abs(d_eff)`.


In [1]:
from __future__ import annotations

import inspect
import math
import os
import shutil
import sys
from pathlib import Path

import numpy as np
from scipy import constants as cs

root = Path.cwd()
if not (root / "src").exists() and (root.parent / "src").exists():
    root = root.parent

mpl_cache = Path("/tmp/matplotlib-arc-rydcalc-cs")
mpl_cache.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(mpl_cache))

# NumPy 2 compatibility for the current rydcalc submodule.
if not hasattr(np, "product"):
    np.product = np.prod
if not hasattr(np, "trapz"):
    np.trapz = np.trapezoid

if str(root / "rydcalc") not in sys.path:
    sys.path.insert(0, str(root / "rydcalc"))

import arc
from arc import Caesium
import arc.alkali_atom_data as arc_atom_data
import rydcalc

# ARC normally copies data into ~/.arc-data. In this sandboxed repo workflow,
# copy the packaged data into /tmp and point Caesium at that cache explicitly.
def prepare_arc_data_cache() -> Path:
    source_candidates = [
        root / "ARC-Alkali-Rydberg-Calculator" / "arc" / "data",
        Path(inspect.getfile(arc)).resolve().parent / "data",
    ]
    source = next(path for path in source_candidates if path.exists())
    target = Path("/tmp/arc-data-cs-60p")
    version = source / "version.txt"
    target_version = target / "version.txt"
    if not target.exists() or (version.exists() and (not target_version.exists() or target_version.read_text() != version.read_text())):
        if target.exists():
            shutil.rmtree(target)
        shutil.copytree(source, target)
    return target

arc_data = prepare_arc_data_cache()
Caesium.dataFolder = str(arc_data)
arc_atom_data.Caesium.dataFolder = str(arc_data)

print(f"project root: {root}")
print(f"ARC version: {getattr(arc, '__version__', 'unknown')}")
print(f"ARC data cache: {arc_data}")
print(f"rydcalc module: {Path(rydcalc.__file__).resolve()}")
print(f"NumPy compatibility aliases: product={hasattr(np, 'product')}, trapz={hasattr(np, 'trapz')}")

project root: /home/eris/projects/Noise-Tolerant-Quantum-Control-Optimization
ARC version: 3.10.2
ARC data cache: /tmp/arc-data-cs-60p
rydcalc module: /home/eris/projects/Noise-Tolerant-Quantum-Control-Optimization/rydcalc/rydcalc/__init__.py
NumPy compatibility aliases: product=True, trapz=True


## Important package calls

ARC calls used here:

- `Caesium()` constructs the Cs atom model.
- `getDipoleMatrixElement(n1, l1, j1, mj1, n2, l2, j2, mj2, q)` returns the `mj`-resolved electric-dipole matrix element in `e a0`.
- `getRadialMatrixElement(...)` and `getReducedMatrixElementJ(...)` expose the radial and reduced-J pieces used by the full angular matrix element.
- `getTransitionWavelength(...)` gives the transition wavelength in meters.

rydcalc calls used here:

- `rydcalc.Cesium133(use_db=False)` constructs the local Cs model without writing a dipole database.
- `atom.get_state((n, l, j, m))` returns a state object for the fine-structure state.
- `state.energy_Hz` is the zero-field energy relative to the ionization limit, in Hz.
- `state.get_multipole_me(other, k=1, qIn=[q])` returns the E1 matrix element for the selected spherical polarization component, again in `e a0`.


In [2]:
transition = {
    "n1": 6,
    "l1": 0,
    "j1": 0.5,
    "mj1": 0.5,
    "n2": 60,
    "l2": 1,
    "j2": 1.5,
    "mj2": 1.5,
    "q": 1,
}

w_x = 10e-6
w_y = 10e-6
target_rabi_hz = [1e6, 5e6, 10e6]
au_dipole_si = cs.e * cs.physical_constants["Bohr radius"][0]


def gaussian_power_for_rabi(rabi_hz: float, d_au: float, w_x: float, w_y: float) -> dict[str, float]:
    d_si = abs(d_au) * au_dipole_si
    omega = 2.0 * math.pi * rabi_hz
    power_w = math.pi * w_x * w_y * cs.c * cs.epsilon_0 * cs.hbar**2 * omega**2 / (4.0 * d_si**2)
    intensity_w_m2 = 2.0 * power_w / (math.pi * w_x * w_y)
    field_v_m = math.sqrt(2.0 * intensity_w_m2 / (cs.c * cs.epsilon_0))
    return {
        "rabi_hz": rabi_hz,
        "power_w": power_w,
        "intensity_w_m2": intensity_w_m2,
        "field_v_m": field_v_m,
    }


def print_power_block(label: str, d_au: float, wavelength_m: float, extra: dict[str, float] | None = None) -> None:
    d_si = d_au * au_dipole_si
    print(label)
    print(f"  lambda = {1e9 * wavelength_m:.6f} nm")
    print(f"  d_eff = {d_au:+.12e} e a0 = {d_si:+.12e} C m")
    if extra:
        for key, value in extra.items():
            print(f"  {key} = {value}")
    print(f"  Gaussian waist: wx = wy = {1e6 * w_x:.1f} um")
    for result in [gaussian_power_for_rabi(rabi, d_au, w_x, w_y) for rabi in target_rabi_hz]:
        print(
            f"  Omega/2pi = {result['rabi_hz'] / 1e6:>4.1f} MHz -> "
            f"P = {result['power_w']:.9f} W = {1e3 * result['power_w']:.6f} mW, "
            f"I0 = {result['intensity_w_m2']:.6e} W/m^2, "
            f"E0 = {result['field_v_m']:.6e} V/m"
        )

In [3]:
arc_cs = Caesium()

arc_dme = arc_cs.getDipoleMatrixElement(**transition)
arc_radial = arc_cs.getRadialMatrixElement(
    transition["n1"], transition["l1"], transition["j1"],
    transition["n2"], transition["l2"], transition["j2"],
)
arc_reduced_j = arc_cs.getReducedMatrixElementJ(
    transition["n1"], transition["l1"], transition["j1"],
    transition["n2"], transition["l2"], transition["j2"],
)
arc_wavelength_m = arc_cs.getTransitionWavelength(
    transition["n1"], transition["l1"], transition["j1"],
    transition["n2"], transition["l2"], transition["j2"],
)
arc_frequency_hz = cs.c / arc_wavelength_m

print_power_block(
    "ARC / Caesium / sigma+",
    arc_dme,
    arc_wavelength_m,
    {
        "radial <nlj|r|n'l'j'> [a0]": f"{arc_radial:+.12e}",
        "reduced-J DME [e a0]": f"{arc_reduced_j:+.12e}",
        "transition frequency [THz]": f"{arc_frequency_hz / 1e12:.6f}",
    },
)

ARC / Caesium / sigma+
  lambda = 318.755379 nm
  d_eff = -1.895269095599e-03 e a0 = -1.606876159714e-32 C m
  radial <nlj|r|n'l'j'> [a0] = +3.282702367592e-03
  reduced-J DME [e a0] = -3.790538191197e-03
  transition frequency [THz] = 940.509486
  Gaussian waist: wx = wy = 10.0 um
  Omega/2pi =  1.0 MHz -> P = 0.000354492 W = 0.354492 mW, I0 = 2.256767e+06 W/m^2, E0 = 4.123572e+04 V/m
  Omega/2pi =  5.0 MHz -> P = 0.008862302 W = 8.862302 mW, I0 = 5.641917e+07 W/m^2, E0 = 2.061786e+05 V/m
  Omega/2pi = 10.0 MHz -> P = 0.035449210 W = 35.449210 mW, I0 = 2.256767e+08 W/m^2, E0 = 4.123572e+05 V/m


In [4]:
ryd_cs = rydcalc.Cesium133(use_db=False)
ryd_initial = ryd_cs.get_state((transition["n1"], transition["l1"], transition["j1"], transition["mj1"]))
ryd_final = ryd_cs.get_state((transition["n2"], transition["l2"], transition["j2"], transition["mj2"]))

ryd_dme = ryd_initial.get_multipole_me(ryd_final, k=1, qIn=[transition["q"]])
ryd_frequency_hz = abs(ryd_final.energy_Hz - ryd_initial.energy_Hz)
ryd_wavelength_m = cs.c / ryd_frequency_hz

print_power_block(
    "rydcalc / Cesium133 / sigma+",
    ryd_dme,
    ryd_wavelength_m,
    {
        "initial state": ryd_cs.repr_state(ryd_initial),
        "final state": ryd_cs.repr_state(ryd_final),
        "transition frequency [THz]": f"{ryd_frequency_hz / 1e12:.6f}",
    },
)

rydcalc / Cesium133 / sigma+
  lambda = 656.514196 nm
  d_eff = -1.297263594371e-02 e a0 = -1.099865949115e-31 C m
  initial state = |Cs:6,S,0.5,0.5>
  final state = |Cs:60,P,1.5,1.5>
  transition frequency [THz] = 456.642765
  Gaussian waist: wx = wy = 10.0 um
  Omega/2pi =  1.0 MHz -> P = 0.000007566 W = 0.007566 mW, I0 = 4.816948e+04 W/m^2, E0 = 6.024434e+03 V/m
  Omega/2pi =  5.0 MHz -> P = 0.000189161 W = 0.189161 mW, I0 = 1.204237e+06 W/m^2, E0 = 3.012217e+04 V/m
  Omega/2pi = 10.0 MHz -> P = 0.000756644 W = 0.756644 mW, I0 = 4.816948e+06 W/m^2, E0 = 6.024434e+04 V/m


In [5]:
arc_power_1mhz = gaussian_power_for_rabi(1e6, arc_dme, w_x, w_y)["power_w"]
ryd_power_1mhz = gaussian_power_for_rabi(1e6, ryd_dme, w_x, w_y)["power_w"]

rows = [
    ("lambda [nm]", 1e9 * arc_wavelength_m, 1e9 * ryd_wavelength_m),
    ("frequency [THz]", arc_frequency_hz / 1e12, ryd_frequency_hz / 1e12),
    ("d_eff [e a0]", arc_dme, ryd_dme),
    ("|d_eff| [e a0]", abs(arc_dme), abs(ryd_dme)),
    ("P at 1 MHz [mW]", 1e3 * arc_power_1mhz, 1e3 * ryd_power_1mhz),
]

print("ARC vs rydcalc direct comparison")
print(f"{'quantity':<22} {'ARC':>18} {'rydcalc':>18} {'rydcalc/ARC':>14}")
for name, arc_value, ryd_value in rows:
    ratio = ryd_value / arc_value if arc_value != 0 else math.nan
    print(f"{name:<22} {arc_value:>18.9e} {ryd_value:>18.9e} {ratio:>14.6f}")

print()
print("Diagnostic conclusion:")
print("  ARC and rydcalc do not agree for the low-lying 6S -> 60P Cs transition in this local setup.")
print("  The rydcalc Cs Rydberg-Ritz model gives a 6S binding energy/transition frequency far from ARC/NIST-based ARC data,")
print("  and its effective DME is larger by a factor of about {:.3f}.".format(abs(ryd_dme) / abs(arc_dme)))
print("  Treat this as a failed low-n validation for rydcalc alkali optical-coupling use, not as a calibrated laser-power estimate.")

ARC vs rydcalc direct comparison
quantity                              ARC            rydcalc    rydcalc/ARC
lambda [nm]               3.187553793e+02    6.565141960e+02       2.059618
frequency [THz]           9.405094861e+02    4.566427654e+02       0.485527
d_eff [e a0]             -1.895269096e-03   -1.297263594e-02       6.844746
|d_eff| [e a0]            1.895269096e-03    1.297263594e-02       6.844746
P at 1 MHz [mW]           3.544920978e-01    7.566444652e-03       0.021344

Diagnostic conclusion:
  ARC and rydcalc do not agree for the low-lying 6S -> 60P Cs transition in this local setup.
  The rydcalc Cs Rydberg-Ritz model gives a 6S binding energy/transition frequency far from ARC/NIST-based ARC data,
  and its effective DME is larger by a factor of about 6.845.
  Treat this as a failed low-n validation for rydcalc alkali optical-coupling use, not as a calibrated laser-power estimate.


## Literature check: which result is more credible?

The literature check supports the ARC result, not the current local `rydcalc.Cesium133` result, for this specific low-lying-to-Rydberg optical transition.

Main points:

- The Cs I ionization limit from the `6S1/2` ground state is about `31406.46769 cm^-1`, corresponding to an ionization-edge wavelength near `318.6 nm`. A `6S1/2 -> 60P3/2` Rydberg transition must therefore also lie close to `319 nm`, slightly below the ionization limit. ARC gives `318.755379 nm`, which matches this physical scale. The current rydcalc result gives `656.514196 nm`, which is not compatible with the Cs ground-to-Rydberg spectrum.
- Deiglmayr et al., Phys. Rev. A 93, 013424 (2016), measured and fitted the Cs `6s1/2 -> np1/2,np3/2` Rydberg series with frequency-comb-calibrated UV spectroscopy. ARC's Cs `S` and `P` quantum-defect data chain is consistent with this kind of NIST/high-resolution-spectroscopy reference path.
- Rydberg-atom quantum-computing and spectroscopy experiments using direct Cs `6S -> nP` excitation use UV light near `319 nm`; for example, single-photon Cs Rydberg spectra around `nP3/2` report a `319 nm` UV laser. This agrees with ARC's wavelength and contradicts the local rydcalc wavelength.
- The likely local rydcalc issue is not a small angular-factor or phase-convention difference. It appears to place the low-lying `6S` reference energy incorrectly for this calculation, probably because the Rydberg-Ritz/quantum-defect path is being used outside its reliable low-`n` regime. That makes the transition frequency wrong by roughly a factor of two, and the derived power wrong by much more because `P ∝ 1 / |d_eff|^2`.

Practical conclusion for this notebook:

- Use ARC as the benchmark for the requested `133Cs |6S1/2, mj=1/2> -> |60P3/2, mj=3/2>` `sigma+` laser-power estimate.
- Treat the current rydcalc value as a failed validation case for low-lying Cs ground-to-Rydberg optical coupling.
- This does not rule out rydcalc for high-Rydberg manifold or pair-interaction work (`C3`, `C6`, Stark maps, etc.), but low-state optical couplings need a NIST/ARC-style low-energy calibration or a separate benchmark before use.

Reference links used for this judgment:

- NIST Cs I data, including the Cs ground-state ionization limit: https://physics.nist.gov/PhysRefData/Handbook/Tables/cesiumtable1_a.htm
- Deiglmayr et al., `Precision measurement of the ionization energy of Cs I`, Phys. Rev. A 93, 013424 (2016): https://doi.org/10.1103/PhysRevA.93.013424
- ARC documentation for alkali energy and dipole-matrix-element APIs: https://arc-alkali-rydberg-calculator.readthedocs.io/en/latest/alkali_atom_functions.html
- ARC Cs data source documentation: https://arc-alkali-rydberg-calculator.readthedocs.io/en/latest/_modules/arc/alkali_atom_data.html
- Bai et al., `Autler-Townes doublet in single-photon Rydberg spectra of cesium atomic vapor with a 319 nm UV laser`, Applied Physics B 125, 33 (2019): https://doi.org/10.1007/s00340-019-7138-3
- Saffman, Walker, and Molmer, `Quantum information with Rydberg atoms`, Rev. Mod. Phys. 82, 2313 (2010): https://doi.org/10.1103/RevModPhys.82.2313


## Interpretation

For the requested `6S1/2 -> 60P3/2` optical transition, ARC and the current local `rydcalc.Cesium133` model are not numerically consistent. ARC gives the expected near-UV transition wavelength around `319 nm`, while rydcalc gives a much lower optical frequency for the same nominal state labels. The literature check above strongly favors ARC for this low-lying-to-Rydberg optical transition.

Adopted estimate for the parameters in this notebook:

```text
|d_eff| = 1.895269095599e-03 e a0
w_x = w_y = 10 um
Omega / (2 pi) = 10 MHz
P = 35.449210 mW
```

The rydcalc result is kept in the notebook because it is the validation target. In this local setup it should be read as a failed low-`n` alkali optical-coupling validation, not as a calibrated laser-power estimate. rydcalc may still be useful for high-Rydberg pair-interaction calculations, but this direct Cs ground-to-Rydberg coupling needs ARC/NIST-style calibration before relying on rydcalc.
